# 01 — Preprocessing Overview

Objectif: documenter **toutes** les étapes de prétraitement du pipeline Raman.

Contenu:
- Chargement des spectres (étalons synthétiques/réels)
- Contrôles qualité (SNR, outliers, NaN)
- Alignement spectral (ancrage pic Si 520 cm⁻¹)
- Correction de ligne de base (ALS/airPLS)
- Lissage (Savitzky–Golay)
- Normalisations (aire, max, z-score)
- Export formaté (Parquet)

## 1) Chargement des données
Utilise `raman_ai.io.standards.load_reference_spectra_mode(root, mode)`.
- `mode='synthetic'` lit `data/standards/*.csv`
- `mode='real'` lit `data/standards/real_templates/*.csv` (+ `.meta.json` si présent)

In [ ]:

import os, sys
sys.path.append(os.path.join("..", "src"))
from raman_ai.io.standards import load_reference_spectra_mode

root = os.path.join("..", "data", "standards")
specs, metas = load_reference_spectra_mode(root, mode="synthetic")
[(s.name, len(s.wavenumber)) for s in specs]


## 2) Contrôle Qualité (QC)
Nous évaluons:
- **Présence de NaN/Inf**
- **Amplitude max/min**
- **SNR** (proxy simple: rapport max/intégrale du bruit haute fréquence)
- **Longueur et monotonie approximative des axes**

In [ ]:

import numpy as np

def qc_basic(wavenumber, intensity):
    wn = np.asarray(wavenumber, float)
    y = np.asarray(intensity, float)
    flags = {}
    flags["has_nan"] = bool(np.isnan(wn).any() or np.isnan(y).any())
    flags["has_inf"] = bool(np.isinf(wn).any() or np.isinf(y).any())
    flags["len_ok"] = (len(wn) >= 5 and len(wn) == len(y))
    # SNR proxy
    if len(y) > 5:
        hf = y - np.convolve(y, np.ones(5)/5.0, mode="same")
        snr = (np.max(y) - np.min(y)) / (np.std(hf) + 1e-12)
    else:
        snr = np.nan
    flags["snr_proxy"] = float(snr)
    # wn monotone non-décroissant
    flags["wn_monotone"] = bool(np.all(np.diff(wn) >= 0))
    return flags

qc = [ (s.name, qc_basic(s.wavenumber, s.intensity)) for s in specs ]
qc[:3]


## 3) Alignement spectral
Ancrage par pic Si **~520 cm⁻¹**: on cherche le maximum près de 520 et on translate l'axe pour aligner exactement à 520.
> Méthode: recherche d'argmax dans [500,540] cm⁻¹, puis `wn_aligned = wn + (520 - wn[argmax])`. 

In [ ]:

def align_to_si520(wavenumber, intensity, window=(500, 540), target=520.0):
    wn = np.asarray(wavenumber, float)
    y = np.asarray(intensity, float)
    mask = (wn >= window[0]) & (wn <= window[1])
    if mask.sum() < 1:
        return wn, y, None
    idx = np.argmax(y[mask])
    peak_wn = wn[mask][idx]
    shift = target - peak_wn
    return wn + shift, y, float(shift)

aligned = []
for s in specs:
    wn_al, y_al, shift = align_to_si520(s.wavenumber, s.intensity)
    aligned.append((s.name, shift))
aligned


## 4) Correction de ligne de base — ALS (Asymmetric Least Squares)
Modèle rapide sans dépendance externe.
Paramètres typiques: `lam=1e5`, `p=0.01`, `niter=10`.

In [ ]:

import numpy as np

def baseline_als(y, lam=1e5, p=0.01, niter=10):
    L = len(y)
    D = np.diff(np.eye(L), 2)
    DTD = D.T @ D
    w = np.ones(L)
    for _ in range(niter):
        W = np.diag(w)
        Z = W + lam * DTD
        b = np.linalg.solve(Z, w * y)
        w = p * (y > b) + (1 - p) * (y <= b)
    return b

# Démo sur un spectre (si peu d'échantillons, fonctionne mais APERÇU)
import matplotlib.pyplot as plt
s0 = specs[0]
y = np.asarray(s0.intensity, float)
b = baseline_als(y, lam=1e4, p=0.01, niter=10)
plt.figure(); plt.plot(s0.wavenumber, y); plt.plot(s0.wavenumber, b); plt.title("Baseline ALS"); plt.xlabel("cm^-1"); plt.ylabel("a.u."); plt.tight_layout(); plt.show()


## 5) Lissage — Savitzky–Golay
Lissage polynomial local (ordre 2 ou 3) avec fenêtre impaire.

In [ ]:

from math import factorial

def savgol(y, window=5, poly=2):
    y = np.asarray(y, float)
    if window % 2 == 0: window += 1
    half = window // 2
    # Construire la matrice de Vandermonde locale
    A = np.vander(np.arange(-half, half+1), N=poly+1, increasing=True)
    # Pseudo-inverse pour le coef central
    ATA = A.T @ A
    inv = np.linalg.pinv(ATA) @ A.T
    coeff = inv[0]  # coefficients du filtre pour la valeur lissée (ordre 0)
    out = np.copy(y)
    for i in range(half, len(y)-half):
        out[i] = np.dot(coeff, y[i-half:i+half+1])
    # bords: copie simple
    out[:half] = y[:half]; out[-half:] = y[-half:]
    return out

y_smooth = savgol(y - b, window=5, poly=2)
plt.figure(); plt.plot(s0.wavenumber, y - b); plt.plot(s0.wavenumber, y_smooth); plt.title("Smoothing (Savitzky–Golay)"); plt.xlabel("cm^-1"); plt.ylabel("a.u."); plt.tight_layout(); plt.show()


## 6) Normalisation
Plusieurs options:
- **max**: `y / max(y)`
- **aire**: `y / ∫ y dν`
- **z-score**: `(y - μ) / σ` (à utiliser avec précaution)

In [ ]:

def normalize(y, mode="max", wn=None):
    y = np.asarray(y, float)
    if mode == "max":
        m = np.max(np.abs(y)) + 1e-12
        return y / m
    elif mode == "area":
        if wn is None:
            return y / (np.trapz(y) + 1e-12)
        else:
            return y / (np.trapz(y, wn) + 1e-12)
    elif mode == "zscore":
        mu, sd = np.mean(y), np.std(y) + 1e-12
        return (y - mu) / sd
    else:
        return y

y_final = normalize(y_smooth, mode="area", wn=np.asarray(s0.wavenumber))
plt.figure(); plt.plot(s0.wavenumber, y_final); plt.title("Final normalized spectrum"); plt.xlabel("cm^-1"); plt.ylabel("a.u."); plt.tight_layout(); plt.show()


## 7) Export
Standardiser l’export en Parquet/CSV avec schéma stable.

In [ ]:

import pandas as pd, os
df = pd.DataFrame({"wavenumber": s0.wavenumber, "intensity": y_final.tolist()})
outp = os.path.join("..", "data", "processed")
os.makedirs(outp, exist_ok=True)
df.to_parquet(os.path.join(outp, f"{s0.name}_preprocessed.parquet"))
df.head()
